In [28]:
import numpy as np
import sklearn
import torch
import os
import pandas as pd
import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchmetrics

### Load the dataset

In [29]:
if not os.path.exists('IMDB-Dataset.csv'):
  !wget -O IMDB-Dataset.csv -q "https://www.dropbox.com/scl/fi/0c7zc2adk1mgwgut5w80w/IMDB-Dataset.csv?rlkey=1drfg4zw36mhu32ndy2ihnygw&dl=1"

In [30]:
df = pd.read_csv('IMDB-Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


####  Preprocess the dataset

In [ ]:
# remove line break tags
text = list(df['review'].str.replace('<br />',''))

# remap labels to 0 (negative) and 1 (positive)
labels = np.array(df['sentiment'].map({'negative':0,'positive':1}))

#### Make train/test split

In [ ]:
from sklearn.model_selection import train_test_split
# 90/10 train/test split
text_train, text_test, labels_train, labels_test = train_test_split(text,labels,test_size=0.1,random_state=42)

### Bag-of-words experiment

Here you can implement the bag-of-words experiment using scikit-learn.

In [ ]:
# YOUR CODE HERE

### RNN Experiment

#### Tokenize the texts

In [33]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# get number of tokens in vocabulary
vocab_size = len(tokenizer.vocab)

In [ ]:
tokenized_text_train = [tokenizer(t)['input_ids'] for t in text_train]
tokenized_text_test = [tokenizer(t)['input_ids'] for t in text_test]

Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors


#### Custom Dataset class

This is a custom subclass of Dataset that will produce sequences of tokens and associated sentiment labels.

If a maximum sequence length is provided, it will randomly truncate the text or zero pad it as necessary.

In [ ]:
class TokenDataset(Dataset):
  def __init__(self,tokenized_text,labels,max_seq_len=None):
    self.tokenized_text = tokenized_text
    self.labels = labels
    self.max_seq_len = max_seq_len
    
  def __len__(self):
    return len(self.tokenized_text)

  def __getitem__(self,idx):
    # get requested text
    token_ids = self.tokenized_text[idx]

    # randomly truncate or zero pad if necessary
    if self.max_seq_len is not None:
      if len(token_ids)>self.max_seq_len:
        # choose random substring
        ind = np.random.randint(len(token_ids)-self.max_seq_len)
        token_ids = token_ids[ind:ind+self.max_seq_len]
      else:
        # pad to maximum sequence length
        token_ids = [0]*(self.max_seq_len-len(token_ids)) + token_ids
    
    # return a sequence of token IDs and a label
    return torch.tensor(token_ids), torch.tensor(self.labels[idx])


In [36]:
train_ds = TokenDataset(tokenized_text_train,labels_train,max_seq_len=100)
test_ds = TokenDataset(tokenized_text_test,labels_test)

train_loader = DataLoader(train_ds,batch_size=32,shuffle=True)
test_loader = DataLoader(test_ds,batch_size=1,shuffle=False)

Here you can implement the RNN experiment.

In [ ]:
# YOUR CODE HERE